# 16 · Collect polygon layers against an independent sort

**Question.** Does Fieldwork's `stack_polygons` widget order polygons the way its declared
rule says, and does it leave them otherwise alone?

This widget does one thing: concatenate layers and order the result largest extent first, so
a smaller band draws on top of a larger one. There is no geometry to compare, so this is a
check of a **rule** rather than of arithmetic.

**Extent is not area**, and that is the thing worth pinning down. It is the sum over every
ring — outer and interior alike — of the absolute shoelace area of that ring, in **square
degrees**, unprojected, with holes added rather than subtracted. It is a sort key and nothing
else. Fieldwork's own comment makes the point: `summarize_hull` and `summarize_combined_area`
take a UTM zone because they report areas, and this widget takes none because it does not.

Criteria fixed before running: the order of identifiers exactly, the same polygons in and out,
and the layer, polygon and banded-polygon counts exact.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["shapely"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_16_stack_polygons import run

report = run(Path.cwd() / "fixtures" / "stack-polygons-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["polygons"], "polygons in", report["layers"], "layers")
print(f"order agrees          {report['orderAgrees']}")
print(f"same polygons in/out  {report['samePolygonsInAndOut']}")
print(f"counts agree          {report['countsAgree']}")
print(f"ties in extent        {report['extentTies']}")
print()
for position, row in enumerate(report["extents"], start=1):
    print(f"  {position}. {row['id']:<28} {row['minutes']:>4} min   {row['extentSquareDegrees']:.3e} sq deg")

## What a tie would mean

Two polygons of equal extent leave the order ambiguous, and a sort is not required to be
stable. The check reports the number of ties for that reason: with none, the expected order is
the only order, and a passing result means what it appears to.

With a tie, this check would still pass while the drawn result could differ between runs — so
the count matters more than it looks.

## Dive deeper · why extent is the wrong number to quote

The rule counts holes **positively** and works in **square degrees**. Both are fine for
sorting and wrong for measuring, and the gap between extent and real area is worth seeing on
the actual bands.

In [ ]:
import shapely
from shapely.geometry import MultiPolygon, Polygon, shape

fixture = json.loads((Path.cwd() / "fixtures" / "stack-polygons-soho-001.json").read_text())

print(f"{'polygon':<28} {'extent sq deg':>14} {'rings':>6} {'holes':>6}")
for layer in fixture["input"]["layers"]:
    for polygon in layer["polygons"]:
        shaped = shape(polygon["geometry"])
        parts = list(shaped.geoms) if isinstance(shaped, MultiPolygon) else [shaped]
        holes = sum(len(part.interiors) for part in parts)
        extent = sum(abs(Polygon(part.exterior).area) for part in parts) + sum(
            abs(Polygon(ring).area) for part in parts for ring in part.interiors)
        net = sum(part.area for part in parts)
        print(f"{polygon['id']:<28} {extent:14.3e} {len(parts):6} {holes:6}")
        print(f"{'  net of holes, sq deg':<28} {net:14.3e}   "
              f"extent exceeds it by {extent / net - 1:.1%}")
print()
print("Both numbers are in square degrees and neither is a measurement of ground. For an area")
print("in square metres, use summarize_combined_area, which takes a UTM zone because it has to.")

### Reading the result

`agrees` means the declared rule, recomputed independently, gives the order Fieldwork
produced. It says nothing about whether largest-first is the right draw order — and it cannot
stop a reader quoting the extent as an area, which is why the rule is declared in full in the
fixture and repeated here.